### RAG PIPELINE (INJESTION + RETRIEVAL)
![Workflow](Images/krish_rag_pipeline.png)


### Here we would be looking at the inference code, how we actually fetch the most relevent chunks from the vectorDB

In [1]:
import os
import uuid
from typing import Any, Dict, List, Tuple
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# 1. Embedding Manager Class
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [ ]:
# 2. Vector Store Class- This code connects to the existing ChromaDB vector store during inference.
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(
                f"Existing documents in collection: {self.collection.count()}"
            )
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

### No add documents present here since this is at the time of inference, not injestion

In [ ]:
# 3. RAG Retriever Class
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self, vector_store: VectorStore, embedding_manager: EmbeddingManager
    ):
        """Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0 # arguments
    ) -> List[Dict[str, Any]]: # returns list of results(strings) and metadata
        """Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0] # use the same model used to embed in injestion time

        # Search in the vector DB
        try:
            # Search in ChromaDB collection
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], # give query embeddings in form of a list
                n_results=top_k # number if top ranked chunks to fetch
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances) # zipping is basically trying to create a tuple here
                ):
                    """
                    # Convert distance to similarity score (ChromaDB uses cosine distance as similarity score)
                    # We dont compare our embedding to ALL the chunks present in the vector DB, ANN - Approximate Nearest Neighbour =  general problem/approach: find nearest vectors approximately without comparing against every vector.
                    # HNSW - Hierarchical Navigable Small World and IVF = Inverted File Index = a specific algorithm/data structure used to solve that ANN problem efficiently. 
                    """

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(
                    f"Retrieved {len(retrieved_docs)} documents (after filtering)"
                )
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [9]:
# Initialize embedding manager
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")

# Connect to your existing saved vector database
# (Verify relative path ../data/vector_store matches where your first notebook created the database)
vector_store = VectorStore(
    collection_name="pdf_documents",
    persist_directory="data/vector_store"
)

# Initialize retriever
retriever = RAGRetriever(
    vector_store=vector_store,
    embedding_manager=embedding_manager
)

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\Nitin Phanse\AppData\Local\Temp\ipykernel_5168\1182172884.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1020


In [10]:
query = "What is attention is all you need"

# Retrieve top 3 relevant chunks
results = retriever.retrieve(query=query, top_k=3, score_threshold=0.0)

# Display retrieved chunks
for doc in results:
    print(f"\n--- Rank {doc['rank']} | Score: {doc['similarity_score']:.4f} ---")
    print(f"Content: {doc['content']}")
    print(f"Metadata: {doc['metadata']}")

Retrieving documents for query: 'What is attention is all you need'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)

--- Rank 1 | Score: 0.1400 ---
Content: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3
Metadata: {'file_type': 'pdf', 'keywords': '', 'creator': 'LaTeX with hyperref', 'author': '', 'doc_index': 165, 'total_pages': 15, 'subject': '', 'page_label': '3', 'title': '', 'source': 'data\\pdf\\attention is all you need.pdf', 'content_length': 216, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source_file': 'attention is all you need.pdf', 'producer': 'pdfTeX-1.40.25', 'page': 2, 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '/False', 'creationdate': '2024-04-10T21:11:43+00:00'}


### What we get above is the chunks, which basically acts as a context for the LLM

## Integration of vectorDB context pipeline with LLM output

In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024,
)
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = (
        "\n\n".join([doc["content"] for doc in results]) if results else ""
    )
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}
    
    Question: {query}
    
    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


# Run the RAG function
answer = rag_simple("What is attention mechanism?", retriever, llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
The attention mechanism is a function that takes a **query** vector and a set of **key‑value** pairs and produces an output vector as a weighted sum of the values. The weights are computed from the similarity between the query and each key, allowing the model to focus on the most relevant parts of the input.


## Advanced RAG, enhanced features

In [ ]:
# --- Enhanced RAG Pipeline Features ---

# 1. THE FUNCTION DEFINITION
# We define our advanced function with adjustable knobs: 
# top_k (how many docs to fetch), min_score (how strict the matching should be), 
# and return_context (a toggle to decide if we want to see the raw text chunks later).
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    
    # --- STEP A: SECURE RETRIEVAL ---
    # Ask the database for the best chunks, but ONLY if they meet the 'min_score' threshold.
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    # If the database returns absolutely nothing that meets our standards, fail gracefully.
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # --- STEP B: PREPARE CONTEXT AND SOURCES ---
    # Glue all the actual text content together so the LLM can read it.
    context = "\n\n".join([doc['content'] for doc in results])
    
    # Dig into the metadata of each document to build a neat list of references.
    sources = [{
        # Try to find 'source_file', if not found try 'source', if neither, write 'unknown'
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        # Try to find the specific page number
        'page': doc['metadata'].get('page', 'unknown'),
        # Record how strong of a match this specific chunk was
        'score': doc['similarity_score'],
        # Grab the first 120 characters to show the user a tiny preview of the text
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    
    # Calculate overall confidence by looking at the highest similarity score in our batch.
    confidence = max([doc['similarity_score'] for doc in results])

    # --- STEP C: GENERATE THE ANSWER ---
    # Construct the exact prompt structure to send to the LLM.
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer: """
    
    # Send the prompt to the LLM (like Groq) and capture its response.
    response = llm.invoke([prompt.format(context=context, query=query)])

    # --- STEP D: PACKAGE AND RETURN ---
    # Neatly bundle the LLM's text, our source list, and our confidence score into a dictionary.
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    # If the user turned the 'return_context' toggle to True, add the raw text to the bundle.
    if return_context:
        output['context'] = context
        
    return output

# --- 2. HOW TO USE IT ---
# Call the function with our question, tools, and specific thresholds.
result = rag_advanced("What is attention mechanism?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

# Print everything out nicely to the terminal!
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

NameError: name 'rag_retriever' is not defined

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    # 1. INITIALIZATION: This runs once when we first set up our pipeline
    def __init__(self, retriever, llm):
        self.retriever = retriever  # The tool that searches our ChromaDB database
        self.llm = llm              # The AI brain (like Groq) that writes the answer
        self.history = []           # A blank list to remember past conversations

    # 2. THE MAIN FUNCTION: This handles the user's question from start to finish
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        
        # --- STEP A: RETRIEVE THE DOCUMENTS ---
        # Ask the database to find the chunks of text most relevant to the question.
        # top_k = how many chunks to bring back. min_score = how closely they must match.
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        # If the database found nothing useful, prepare a generic failure message
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # If we found matches, glue all the text chunks together separated by two empty lines
            context = "\n\n".join([doc['content'] for doc in results])
            
            # Create a tidy list containing exactly where this info came from
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...' # A tiny 120-character sneak peek
            } for doc in results]

        # --- STEP B: GENERATE THE ANSWER ---
        # Build the exact instructions for the AI, combining the found context and the question
        prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: """
        
        # This block creates a 'typewriter' effect in the terminal 
        if stream:
            print("Streaming answer:")
            # It prints 80 characters at a time, pausing for 0.05 seconds in between
            for i in range(0, len(prompt), 80):
                print(prompt[i:i+80], end='', flush=True)
                time.sleep(0.05)
            print()
        
        # Send the finalized prompt to the AI model and get the generated text back
        response = self.llm.invoke([prompt])
        answer = response.content

        # --- STEP C: ADD CITATIONS ---
        # Create little reference tags (e.g., "[1] textbook.pdf (page 5)")
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        
        # Stick those citation tags at the very bottom of the AI's final answer
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # --- STEP D: SUMMARIZE (Optional) ---
        summary = None
        # If the user turned on 'summarize', ask the AI a SECOND time to compress its own answer
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # --- STEP E: SAVE MEMORY ---
        # Save everything that just happened into our history list so we don't forget it later
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        # Finally, hand all this beautifully packaged data back to the user
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# --- 3. HOW TO USE THE PIPELINE ---
# Initialize our advanced class using the retriever and LLM we set up earlier
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

# Ask a question and turn on our fancy new 'stream' and 'summarize' toggles
result = adv_rag.query("What is positional encoding?", top_k=3, min_score=0.3, stream=True, summarize=True)

# Print out the final results
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])